# Trackastra Investigation 01 — Model and Environment

## Purpose

This notebook establishes the exact software environment and pretrained
Trackastra model used for the Biohub cell-tracking investigation.

The goals are to:

1. Record the Python, PyTorch, CUDA, and Trackastra environment.
2. Confirm that Trackastra runs on the NVIDIA GPU.
3. Load the pretrained `ctc` model.
4. Inspect the model configuration and architecture.
5. Determine the model's expected input/output conventions.
6. Record enough information to make later experiments reproducible.

No Biohub data is processed in this notebook.

## 1. Environment Information

First, record the exact Python and package versions used by this experiment.

In [ ]:
import sys
import platform
import importlib.metadata as metadata

import torch
import torchvision
import numpy as np

print("Python")
print("------")
print("Version:", sys.version.replace("\n", " "))
print("Executable:", sys.executable)
print("Platform:", platform.platform())

print("\nCore packages")
print("-------------")
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)

for package in [
    "trackastra",
    "geff",
    "motile",
    "scikit-image",
    "scipy",
    "pandas",
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

## 2. CUDA and GPU Verification

Verify that the Jupyter kernel can access the CUDA-enabled PyTorch installation
and that Trackastra can use the NVIDIA GPU.

This check is performed independently of Trackastra so that GPU/environment
problems can be distinguished from model-specific problems.

In [ ]:
import torch

print("## CUDA")
print()

print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    device_index = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_index)

    print()
    print("## GPU")
    print()
    print("Current device index:", device_index)
    print("Device name:", torch.cuda.get_device_name(device_index))
    print(f"Total VRAM: {props.total_memory / 1024**3:.2f} GB")
    print("Compute capability:", f"{props.major}.{props.minor}")

    # Small CUDA computation to verify actual execution.
    x = torch.randn((1024, 1024), device="cuda")
    y = x @ x

    print()
    print("## CUDA computation test")
    print()
    print("Result device:", y.device)
    print(f"Allocated VRAM: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

    del x, y
    torch.cuda.empty_cache()
else:
    print("\nERROR: CUDA is not available to this notebook kernel.")

## 3. Load the Pretrained Trackastra CTC Model

Load the pretrained `ctc` Trackastra model on the CUDA device.

The checkpoint has already been downloaded to the local Trackastra cache, so
this step should load the existing model without downloading it again.

At this stage, only basic model metadata and GPU memory usage are inspected.
The internal transformer architecture and configuration are examined separately
in the following sections.

In [ ]:
import time
import torch

from trackastra.model import Trackastra


# Start from a clean CUDA cache so that the memory measurements are easier
# to interpret.
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

memory_before = torch.cuda.memory_allocated()
reserved_before = torch.cuda.memory_reserved()

start_time = time.perf_counter()

model = Trackastra.from_pretrained(
    "ctc",
    device="cuda",
)

load_time = time.perf_counter() - start_time

memory_after = torch.cuda.memory_allocated()
reserved_after = torch.cuda.memory_reserved()
peak_memory = torch.cuda.max_memory_allocated()


print("## Trackastra model")
print()

print("Wrapper class:", type(model).__name__)
print("Full class:", f"{type(model).__module__}.{type(model).__name__}")

print()
print("## Basic attributes")
print()

for attr in [
    "device",
    "batch_size",
]:
    if hasattr(model, attr):
        print(f"{attr}: {getattr(model, attr)}")
    else:
        print(f"{attr}: <not exposed as a public attribute>")

print()
print("## Loading")
print()

print(f"Load time: {load_time:.2f} s")

print()
print("## CUDA memory")
print()

print(f"Allocated before load: {memory_before / 1024**2:.2f} MB")
print(f"Allocated after load:  {memory_after / 1024**2:.2f} MB")
print(f"Model allocation delta: {(memory_after - memory_before) / 1024**2:.2f} MB")
print(f"Reserved after load:   {reserved_after / 1024**2:.2f} MB")
print(f"Peak allocated:        {peak_memory / 1024**2:.2f} MB")

print()
print("## Wrapper attributes")
print()

print(sorted(model.__dict__.keys()))

## 4. Pretrained Model Configuration

Trackastra stores the training/model configuration associated with the
pretrained checkpoint in `train_args`.

This configuration is inspected before examining the network architecture
because it describes important assumptions such as temporal context,
feature dimensions, spatial encoding, augmentation, and transformer settings.

The configuration printed here belongs to the actual installed `ctc`
checkpoint rather than assumptions taken from the publication.

In [ ]:
from pprint import pprint
from dataclasses import asdict, is_dataclass


train_args = model.train_args

print("## train_args")
print()
print("Type:", type(train_args))
print()


# Convert the configuration into a normal dictionary when possible.
if isinstance(train_args, dict):
    train_config = train_args

elif is_dataclass(train_args):
    train_config = asdict(train_args)

elif hasattr(train_args, "__dict__"):
    train_config = vars(train_args)

else:
    train_config = None


if train_config is not None:
    print(f"Number of configuration entries: {len(train_config)}")
    print()

    for key in sorted(train_config):
        value = train_config[key]
        print(f"{key}: {value}")

else:
    print("Could not automatically convert train_args to a dictionary.")
    print()
    pprint(train_args)

## 5. Instantiated Model Architecture

Inspect the two major components exposed by the Trackastra wrapper:

- `feature_extractor`
- `transformer`

The pretrained `ctc` checkpoint reports `features="wrfeat"`, so a separate
deep image feature extractor is not expected. The transformer configuration
and parameter counts are inspected directly from the loaded model.

In [ ]:
import torch


print("## Wrapper components")
print()

print("Feature extractor:")
print("  value:", model.feature_extractor)
print("  type: ", type(model.feature_extractor))

print()

print("Transformer:")
print("  type: ", type(model.transformer))
print("  module:", type(model.transformer).__module__)


print()
print("## Transformer configuration")
print()

transformer_config = model.transformer.config

for key in sorted(transformer_config):
    print(f"{key}: {transformer_config[key]}")


print()
print("## Parameter counts")
print()

total_params = sum(
    p.numel()
    for p in model.transformer.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.transformer.parameters()
    if p.requires_grad
)

parameter_memory_fp32 = total_params * 4 / 1024**2

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Approx. FP32 size:    {parameter_memory_fp32:.2f} MB")

In [ ]:
print("## Major component parameter counts")
print()

components = {
    "input projection": model.transformer.proj,
    "encoder": model.transformer.encoder,
    "decoder": model.transformer.decoder,
    "head_x": model.transformer.head_x,
    "head_y": model.transformer.head_y,
}

if hasattr(model.transformer, "feat_embed"):
    components["feature embedding"] = model.transformer.feat_embed

if hasattr(model.transformer, "pos_embed"):
    components["position embedding"] = model.transformer.pos_embed


for name, component in components.items():
    n_params = sum(p.numel() for p in component.parameters())
    print(f"{name:20s}: {n_params:>12,d}")

## 6. Cell Feature Representation

The pretrained `ctc` model does not use a separate deep image feature
extractor. Instead, each segmented instance is converted into region-based
features together with its centroid.

A small synthetic 3D example is used here to verify the exact feature names,
dimensions, coordinate ordering, and final feature-vector size produced by the
installed Trackastra version.

No Biohub data is used in this test.

In [ ]:
import numpy as np

from trackastra.data.wrfeat import get_features


# ---------------------------------------------------------------------
# Create a minimal synthetic 3D sequence
# shape = (T, Z, Y, X)
# ---------------------------------------------------------------------

T, Z, Y, X = 4, 16, 32, 32

imgs_test = np.zeros((T, Z, Y, X), dtype=np.float32)
masks_test = np.zeros((T, Z, Y, X), dtype=np.int32)


for t in range(T):
    # Cell 1
    masks_test[
        t,
        4:8,
        8 + t:14 + t,
        8:14,
    ] = 1

    # Cell 2
    masks_test[
        t,
        9:13,
        18:24,
        18 - t:24 - t,
    ] = 2

    # Give the cells different intensities
    imgs_test[t][masks_test[t] == 1] = 0.6
    imgs_test[t][masks_test[t] == 2] = 0.9


print("Image shape:", imgs_test.shape)
print("Mask shape: ", masks_test.shape)
print("Image dtype:", imgs_test.dtype)
print("Mask dtype: ", masks_test.dtype)

In [ ]:
features_test = get_features(
    detections=masks_test,
    imgs=imgs_test,
    features="wrfeat",
    ndim=3,
    n_workers=0,
)

print("Number of frames:", len(features_test))

f0 = features_test[0]

print()
print("## First-frame detections")
print()

print("Labels:", f0.labels)
print("Coordinates shape:", f0.coords.shape)
print("Coordinates (Z, Y, X):")
print(f0.coords)

print()
print("## Feature components")
print()

total_feature_dim = 0

for name, values in f0.features.items():
    print(f"{name:28s} shape={values.shape}")
    total_feature_dim += values.shape[-1]

print()
print("Total shallow feature dimension:", total_feature_dim)
print("Stacked feature shape:", f0.features_stacked.shape)

## Conclusions

The Trackastra `ctc` pretrained model is successfully installed and runs on
the existing CUDA-enabled PyTorch environment.

### Environment

- Python: 3.11.9
- PyTorch: 2.13.0+cu130
- Torchvision: 0.28.0+cu130
- Trackastra: 0.5.5
- GPU: NVIDIA GeForce RTX 4050 Laptop GPU
- GPU memory: 6 GB

### Pretrained model

- Model: `ctc`
- Native dimensionality: 3D
- Temporal window: 4 frames
- Transformer dimension: 512
- Attention heads: 4
- Encoder layers: 6
- Decoder layers: 6
- Parameters: 27,456,880
- Runtime batch size on this GPU: 4

### Cell representation

The `ctc` checkpoint uses `wrfeat` region features rather than a separate
deep image feature extractor.

For each segmented instance, Trackastra extracts:

- centroid `(Z, Y, X)`
- equivalent diameter
- mean intensity
- 3D inertia tensor
- border distance

This produces 12 shallow region features per detection.

### Important observation

Trackastra uses centroid coordinates directly in positional encoding,
attention, spatial-distance gating, and association normalization.

The current implementation does not appear to apply physical voxel spacing
when computing these coordinates or distances.

Therefore, spatial anisotropy must be investigated before interpreting
zero-shot tracking performance on the Biohub data.